# Cartoon Face Mask

## TASK: Cartoonify faces in video feed from live webcam

### Steps
- 1. **Capture video** feed from webcam
- 2. **Recognize faces** in the video
- 3. **Replace/Mask the face** region with your favorite cartoon character
- 4. **Save the video** feed into a video file

### Helper code to recognize faces

In [4]:
#!pip uninstall opencv-python -y
#!pip install opencv-contrib-python==5.0.0.93


In [1]:
import urllib.request
import os

os.makedirs('cascades', exist_ok=True)

files = {
    'haarcascade_frontalface_alt2.xml': 
        'https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_alt2.xml',
    'haarcascade_eye_tree_eyeglasses.xml':
        'https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_eye_tree_eyeglasses.xml'
}

for filename, url in files.items():
    path = os.path.join('cascades', filename)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
        print(f'Downloaded: {filename}')
    else:
        print(f'Already exists: {filename}')

Downloaded: haarcascade_frontalface_alt2.xml
Downloaded: haarcascade_eye_tree_eyeglasses.xml


In [1]:
import cv2
import os

cascPathface = 'cascades/haarcascade_frontalface_alt2.xml'
cascPatheyes = 'cascades/haarcascade_eye_tree_eyeglasses.xml'

faceCascade = cv2.CascadeClassifier(cascPathface)
eyeCascade = cv2.CascadeClassifier(cascPatheyes)

cartoon_img = cv2.imread('Mickey_face.png', cv2.IMREAD_UNCHANGED)

def overlay_image(background, overlay, x, y, w, h):
    bg_h, bg_w = background.shape[:2]
    if x < 0 or y < 0 or x + w > bg_w or y + h > bg_h:
        return background

    overlay_resized = cv2.resize(overlay, (w, h))

    if overlay_resized.shape[2] == 4:
        overlay_rgb = overlay_resized[:, :, :3]
        alpha_mask = overlay_resized[:, :, 3] / 255.0
        roi = background[y:y+h, x:x+w]
        for c in range(3):
            roi[:, :, c] = (alpha_mask * overlay_rgb[:, :, c] +
                             (1 - alpha_mask) * roi[:, :, c])
        background[y:y+h, x:x+w] = roi
    else:
        background[y:y+h, x:x+w] = overlay_resized

    return background

video_capture = cv2.VideoCapture(0)

frame_width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = video_capture.get(cv2.CAP_PROP_FPS)
if fps == 0 or fps != fps:
    fps = 20.0

fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, fps, (frame_width, frame_height))

while True:
    ret, frame = video_capture.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = faceCascade.detectMultiScale(gray,
                                          scaleFactor=1.1,
                                          minNeighbors=5,
                                          minSize=(60, 60),
                                          flags=cv2.CASCADE_SCALE_IMAGE)
    for (x, y, w, h) in faces:
        frame = overlay_image(frame, cartoon_img, x, y, w, h)

    out.write(frame)
    cv2.imshow('Face Video', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    if cv2.getWindowProperty('Face Video', cv2.WND_PROP_VISIBLE) < 1:
        break

video_capture.release()
out.release()
cv2.destroyAllWindows()
for i in range(5):
    cv2.waitKey(1)